# Bài 7 - Mini Project: Chatbot (Phần 1)

## Mục tiêu notebook

Sau notebook này, học viên có thể:

1. Thiết lập Gemini API key an toàn bằng biến môi trường `GEMINI_API_KEY`.
2. Gọi Gemini API bằng Google GenAI SDK mới.
3. Thêm ngữ cảnh/system instruction cho chatbot trợ lý nhà hàng.
4. Viết hàm hỏi - đáp cơ bản để chuẩn bị xây dựng giao diện Streamlit ở Buổi 8.

> Lưu ý hotfix: Notebook này dùng SDK mới `google-genai`, thay cho SDK cũ `google-generativeai`.

## 0. Chuẩn bị API key

### Cách lấy API key

1. Truy cập Google AI Studio.
2. Vào mục **API Keys**.
3. Chọn **Create API Key**.
4. Copy API key và lưu vào file `.env`.

File `.env` nên có dạng:

```bash
GEMINI_API_KEY=your_api_key_here
```

### Quy tắc bảo mật

- Không dán API key trực tiếp vào code.
- Không gửi API key lên GitHub.
- Không chia sẻ API key trong nhóm chat/lớp học.
- Khi deploy Streamlit, dùng **Secrets** thay vì upload file `.env`.

Nếu chưa tạo được API key, notebook vẫn có **mock mode** để học tiếp luồng chatbot.

In [38]:
# Cài thư viện cần thiết
# Chạy cell này nếu môi trường của bạn chưa có các thư viện bên dưới.

%pip install -q -U google-genai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [40]:
# Load biến môi trường từ file .env nếu có
load_dotenv()

MODEL_NAME = "gemini-3.6-flash"

# Nếu USE_MOCK = True, notebook sẽ không gọi API thật.
# Dùng khi học viên chưa tạo được API key hoặc API key bị lỗi.
USE_MOCK = False

print("Đã import thư viện và thiết lập model:", MODEL_NAME)

Đã import thư viện và thiết lập model: gemini-3.6-flash


In [41]:
def load_api_key():
    """Đọc Gemini API key từ biến môi trường.

    TODO:
    - Điền tên biến môi trường chuẩn mới: GEMINI_API_KEY
    - Có thể giữ GOOGLE_API_KEY làm fallback nếu cần
    """
    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    return api_key


api_key = load_api_key()

if api_key:
    print("Đã tìm thấy API key.")
else:
    print("Chưa tìm thấy API key. Có thể bật USE_MOCK = True để học tiếp.")

Đã tìm thấy API key.


In [42]:
# Để gửi yêu cầu đến máy chủ Google, chúng ta cần một đối tượng trung gian gọi là 'Client' (Khách hàng / Đại diện kết nối). 
# Đối tượng này cầm theo API Key để Google xác thực danh tính.
client = None

if USE_MOCK:
    print("Đang bật mock mode. Notebook sẽ không gọi Gemini API thật.")
elif not api_key:
    USE_MOCK = True
    print("Không có API key. Tự động chuyển sang mock mode.")
else:
    # TODO: Tạo Gemini client bằng API key đã load.
    client = genai.Client(api_key=api_key)
    print("Đã tạo Gemini client thành công.")

Đã tạo Gemini client thành công.


In [43]:
# Đây là 'trái tim' của toàn bộ ứng dụng. Hàm này chịu trách nhiệm đóng gói câu hỏi của người dùng, 
# gắn kèm chỉ thị hệ thống, gửi tới Gemini, và bọc trong khối try...except để ứng dụng không bao giờ bị 'sập' nếu mất mạng.

def mock_generate_text(prompt, system_instruction=None):
    """Phản hồi giả lập để lớp học không bị kẹt nếu API key lỗi."""
    return (
        "Xin chào, mình là PhoBot. Đây là phản hồi mẫu trong mock mode. "
        "Khi có API key hợp lệ, phản hồi này sẽ được thay bằng câu trả lời từ Gemini."
    )


def generate_text(prompt, system_instruction=None):
    """Gọi Gemini để sinh câu trả lời từ prompt."""
    if USE_MOCK or client is None:
        return mock_generate_text(prompt, system_instruction)

    config = None
    if system_instruction:
        # TODO: Tạo GenerateContentConfig với system_instruction.
        # `types.GenerateContentConfig(system_instruction=...)`: 
        # Lớp đóng gói cấu hình của SDK. Nó cho phép truyền 'luật chơi' (system instruction) cho mô hình AI.
        config = types.GenerateContentConfig(
            system_instruction=system_instruction
        )
    
    """Bọc phần gọi API bằng try/except để fallback mock hoặc hiển thị lỗi thân thiện:"""
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=config,
        )
    except Exception as error:
        print("Không gọi được Gemini API. Chuyển sang mock response.")
        print("Lỗi:", error)
        return mock_generate_text(prompt, system_instruction) 
        

    return response.text

In [46]:
# TODO: Viết một prompt ngắn để kiểm tra chatbot.
test_prompt = "Hãy giới thiệu về THPT chuyên Lê Hồng Phong, TP.HCM"

answer = generate_text(test_prompt)
print(answer)

**Trường THPT Chuyên Lê Hồng Phong (TP.HCM)** là một trong những ngôi trường trung học phổ thông nổi tiếng, lâu đời và có chất lượng giáo dục hàng đầu tại Việt Nam. Được xem là "cánh chim đầu đàn" của ngành giáo dục miền Nam, trường không chỉ là nơi đào tạo ra vô số nhân tài cho đất nước mà còn là một biểu tượng kiến trúc, lịch sử của TP.HCM.

Dưới đây là thông tin chi tiết về trường THPT Chuyên Lê Hồng Phong:

---

### 1. Lịch sử hình thành và phát triển
* **Thành lập:** Trường được thành lập vào năm **1927** với tên gọi ban đầu là **Trường Trung học Pétrus Trương Vĩnh Ký** (thường gọi là Trường Pétrus Ký). Đây là một trong những trường trung học đầu tiên được Pháp xây dựng tại Sài Gòn.
* **Đổi tên:** Sau ngày đất nước thống nhất, năm 1976, trường vinh dự được mang tên Tổng Bí thư **Lê Hồng Phong**.
* **Mô hình chuyên:** Từ năm 1995, trường chính thức trở thành **Trường THPT Chuyên Lê Hồng Phong**, có nhiệm vụ trọng tâm là phát hiện, bồi dưỡng và đào tạo học sinh giỏi, năng khiếu cho 

In [47]:
MENU_PATH = Path("menu.csv")

if MENU_PATH.exists():
    menu_df = pd.read_csv('menu.csv')
else:
    # Fallback để notebook vẫn chạy nếu học viên mở riêng file notebook.
    menu_df = pd.DataFrame(
        [
            {
                "name": "Pho Bo",
                "description": "Phở bò truyền thống với nước dùng thanh, thịt bò và bánh phở.",
                "price": 12,
            },
            {
                "name": "Bun Cha",
                "description": "Bún chả Hà Nội gồm thịt nướng, bún, rau sống và nước chấm.",
                "price": 11,
            },
            {
                "name": "Goi Cuon",
                "description": "Gỏi cuốn tươi với tôm, rau, bún và nước chấm.",
                "price": 8,
            },
        ]
    )

# TODO: Quan sát 5 dòng đầu tiên của menu.
menu_df.head()

,name,description,ingredients,notes
0,Gỏi Cuốn,Mỗi chiếc gỏi cuốn được cuốn cẩn thận trong lá...,"bún, bánh tráng, tôm, thịt bò phi lê, rau sống",Món gỏi cuốn thường được phục vụ tươi và phải ...
1,Phở Việt Nam,Nổi tiếng với hương vị đậm đà và hương thơm củ...,"bún phở, thịt bò, thịt gà, hành tây, hành phi,...",Thịt bò có thể chọn giữa tái và chín.
2,Cơm Tấm,Cơm tấm là một món ăn đường phố phổ biến trong...,"gạo tấm, thịt heo, trứng, chả, dưa leo, nước m...",Cơm tấm thường được ăn vào bữa trưa hoặc bữa t...
3,Bún Bò,Bún bò là một món ăn đặc trưng của ẩm thực miề...,"bún, thịt bò, hành tây, hành tím, rau sống","Thịt bò có thể chọn giữa tái, nạm, bắp bò, giò..."
4,Khoai Tây Chiên,Khoai tây chiên là một món ăn phổ biến và được...,"khoai tây, dầu, muối",NaN


In [48]:
# Gemini LLM không hiểu được trực tiếp một object bảng dữ liệu DataFrame của Python. AI chỉ hiểu văn bản tự nhiên (Text). 
# Hàm này có nhiệm vụ 'dịch' từng dòng trong bảng thành các câu văn mô tả mạch lạc, chuẩn ngữ pháp tiếng Việt để nạp vào prompt.

def build_menu_context(menu_df):
    """Chuyển dữ liệu menu thành đoạn ngữ cảnh dạng text cho chatbot."""
    def clean_text(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    def add_detail(item_text, label, value):
        value = clean_text(value)
        if not value:
            return item_text

        suffix = "" if value.endswith((".", "!", "?")) else "."
        return f"{item_text} {label}: {value}{suffix}"

    lines = []

    for _, row in menu_df.iterrows():
        # TODO: Lấy tên món, mô tả, thành phần và ghi chú từ từng dòng dữ liệu.
        name = clean_text(row.get("name", ""))
        description = clean_text(row.get("description", ""))
        ingredients = row.get("ingredients", "")
        notes = row.get("notes", "")

        # TODO: Ghép thông tin thành một dòng dễ đọc cho chatbot.
        item_text = f"- {name}: {description}"
        item_text = add_detail(item_text, "Thành phần", ingredients)
        item_text = add_detail(item_text, "Ghi chú", notes)

        lines.append(item_text)

    return "\n".join(lines)


menu_context = build_menu_context(menu_df)
print(menu_context[:800])

- Gỏi Cuốn: Mỗi chiếc gỏi cuốn được cuốn cẩn thận trong lá bánh tráng mềm mại, kết hợp với các loại rau sống, thịt hoặc tôm tươi, và thường được thưởng thức cùng với nước mắm pha chua ngọt. Thành phần: bún, bánh tráng, tôm, thịt bò phi lê, rau sống. Ghi chú: Món gỏi cuốn thường được phục vụ tươi và phải ăn liền sau khi cuốn để trải nghiệm hương vị tốt nhất.
- Phở Việt Nam: Nổi tiếng với hương vị đậm đà và hương thơm của gia vị tự nhiên. mỗi tô phở được chế biến từ nước dùng thơm ngon, kết hợp với sợi bún mềm mại và các loại thịt hoặc hải sản tươi ngon. Thành phần: bún phở, thịt bò, thịt gà, hành tây, hành phi, rau sống, giá đỗ, ớt. Ghi chú: Thịt bò có thể chọn giữa tái và chín.
- Cơm Tấm: Cơm tấm là một món ăn đường phố phổ biến trong ẩm thực Việt Nam. Mỗi đĩa cơm tấm thường bao gồm cơm nấ


In [50]:
system_instruction = """
Bạn tên là PhoBot, một trợ lý AI hỗ trợ khách hàng của nhà hàng Viet Cuisine.

Nhiệm vụ của bạn:
1. Giới thiệu ngắn gọn về nhà hàng Viet Cuisine.
2. Trả lời các câu hỏi liên quan đến menu, món ăn, thành phần và ghi chú món.
3. Gợi ý món ăn phù hợp dựa trên nhu cầu của khách hàng.

Quy tắc trả lời:
- Trả lời bằng tiếng Việt, thân thiện và lịch sự.
- Không bịa thông tin ngoài dữ liệu được cung cấp.
- Nếu khách hỏi ngoài phạm vi nhà hàng/menu, hãy nói:
  "Mình chưa hỗ trợ thông tin này. Bạn vui lòng liên hệ nhân viên nhà hàng để được hỗ trợ thêm."
- Câu trả lời nên ngắn gọn, dễ hiểu.
"""

print(system_instruction)


Bạn tên là PhoBot, một trợ lý AI hỗ trợ khách hàng của nhà hàng Viet Cuisine.

Nhiệm vụ của bạn:
1. Giới thiệu ngắn gọn về nhà hàng Viet Cuisine.
2. Trả lời các câu hỏi liên quan đến menu, món ăn, thành phần và ghi chú món.
3. Gợi ý món ăn phù hợp dựa trên nhu cầu của khách hàng.

Quy tắc trả lời:
- Trả lời bằng tiếng Việt, thân thiện và lịch sự.
- Không bịa thông tin ngoài dữ liệu được cung cấp.
- Nếu khách hỏi ngoài phạm vi nhà hàng/menu, hãy nói:
  "Mình chưa hỗ trợ thông tin này. Bạn vui lòng liên hệ nhân viên nhà hàng để được hỗ trợ thêm."
- Câu trả lời nên ngắn gọn, dễ hiểu.



In [51]:
def ask_bot(question):
    """Tạo prompt có ngữ cảnh menu và gọi Gemini/mock để trả lời."""
    # TODO: Ghép menu_context và question vào user_prompt.
    user_prompt = f"""
Dưới đây là thông tin menu của nhà hàng Viet Cuisine:

{menu_context}

Câu hỏi của khách hàng:
{questions}
"""

    # TODO: Gọi hàm generate_text với prompt và system_instruction.
    return generate_text(
        prompt=user_prompt,
        system_instruction=system_instruction,
    )

In [ ]:
# TODO: Thêm ít nhất 3 câu hỏi để kiểm tra chatbot.
questions = [
    "___",
    "___",
    "___",
]

for question in questions:
    print("Khách hàng:", question)
    print("PhoBot:", ask_bot(question))
    print("-" * 80)

## Checklist cuối buổi

Học viên hoàn thành notebook khi:

- [ ] Hiểu API key là gì và vì sao cần bảo mật.
- [ ] Biết lưu API key bằng `GEMINI_API_KEY`.
- [ ] Biết tạo Gemini client bằng Google GenAI SDK.
- [ ] Gọi được Gemini API hoặc chạy được mock mode.
- [ ] Tạo được system instruction cho chatbot nhà hàng.
- [ ] Viết được hàm `ask_bot(question)`.
- [ ] Test chatbot với ít nhất 3 câu hỏi.

## Chuẩn bị cho Buổi 8

Ở Buổi 8, phần logic chatbot này sẽ được đưa vào giao diện Streamlit bằng:

- `st.chat_input`
- `st.chat_message`
- `st.session_state`